In [ ]:
# report_file_path = "/home/kurt/Downloads/system_orig2.json"
report_file_path = "../f1tenth_drive/results/safe_system/system.json"

In [ ]:
import matplotlib
matplotlib.rcParams.update({'font.size': 22})

In [ ]:
import json
with open(report_file_path, "r") as f:
    report_data = json.load(f)

In [ ]:
print(len(report_data))

In [ ]:
from dataclasses import dataclass
@dataclass
class Record:
    node_name: str
    start_time: float
    end_time: float
    cpu_time: float
    mode:str = 'lo'

@dataclass
class ModeSwitch:
    time: float
    mode: str

# each line from the json has a time - get the minimum time to normalize timestamps
min_time = min(entry["time"] for entry in report_data)
# normalize timestamps
for entry in report_data:
    entry["time"] -= min_time
    entry["time"] = entry["time"] / 1e6  # convert to seconds


# and sort by time
report_data.sort(key=lambda x: x["time"])


current_mode = 'lo'
records = []
mode_switches = []
start_records = {}
for line in report_data:
    time, entry = line["time"], line["entry"]
    if entry["type"] == "exec_start":
        start_records[entry["desc"]] = line
    elif entry["type"] == "exec_end":
        start_entry = start_records.pop(entry["desc"], None)
        if start_entry is not None:
            record = Record(
                node_name=entry["desc"].split(":")[0],
                start_time=start_entry["time"],
                end_time=time,
                cpu_time=entry["cpu_time_us"],
            )
            record.mode = current_mode
            records.append(record)
        else:
            print(f"Warning: No start record for {entry['desc']}")
    elif entry["type"] == "switch_hi":
        current_mode = 'hi'
        mode_switches.append(ModeSwitch(time=time, mode='hi'))
    elif entry["type"] == "switch_lo":
        current_mode = 'lo'
        mode_switches.append(ModeSwitch(time=time, mode='lo'))
    else:
        print(f"Unknown entry type: {entry['type']}")


print(f"Total records: {len(records)}")
print(f"Total mode switches: {len(mode_switches)}")

In [ ]:
# make a swimlane plot of the records
import matplotlib.pyplot as plt

# use interactive backend
%matplotlib widget

# timeline_start_time = 46.2
# timeline_stop_time = 47

timeline_start_time = 5
timeline_stop_time = 12

# swimlane plot
name_to_id = {name: i for i, name in enumerate(sorted(list(set([record.node_name for record in records]))))}
print("default name to id: ", name_to_id)
natural_names = ["simple_timer", "mpc_timer", "mpc_early_stop_sub", "dummy0_timer", "dummy1_timer", 
                 "mpc_car_state_sub", "mpc_local_map_sub", "mpc_user_command_sub", 
                 "simple_car_state_sub", "simple_local_map_sub", "simple_user_command_sub"]
natural_name_to_id = {name: i for i, name in enumerate(natural_names)}
# natural_name_to_id = {"simple_timer": 0, "mpc_timer": 1, "mpc_early_stop_sub": 2, "dummy0_timer": 3, "dummy1_timer": 4, "mpc_car_state_sub": 5
print("initial adjustment: ", natural_name_to_id)
# place missing name_to_id values in natural_name_to_id
# for name in name_to_id.keys():
#     if name not in natural_name_to_id:
#         natural_name_to_id[name] = len(natural_name_to_id)
name_to_id = natural_name_to_id
print("adjusted name to id: ", name_to_id)

name_to_friendly = {"simple_timer": r"$\tau_{FB}$", "mpc_timer": r"$\tau_{MPC}$", "mpc_early_stop_sub": r"$\tau_{MPC}$ Early Stop", 
                    "dummy0_timer": "Dummy0", "dummy1_timer": "Dummy1",
                    "mpc_car_state_sub": r"$\tau_{MPC}$ Car State", "mpc_local_map_sub": r"$\tau_{MPC}$ Env. Update",
                    "mpc_user_command_sub": r"$\tau_{MPC}$ Input",
                    "simple_car_state_sub": r"$\tau_{FB}$ Car State", "simple_local_map_sub": r"$\tau_{FB}$ Env. Update",
                    "simple_user_command_sub": r"$\tau_{FB}$ Input"}

fig, ax = plt.subplots(figsize=(10, 6))
for record in records:
    if record.node_name not in name_to_id:
        continue
    if record.end_time < timeline_start_time or record.start_time > timeline_stop_time:
        continue
    lane = name_to_id[record.node_name]
    ax.broken_barh(
        [(record.start_time, record.end_time - record.start_time)],
        (lane - 0.4, 0.8),
        facecolors='tab:blue',
        edgecolors='black',
        linewidths=1.0
    )

for switch in mode_switches:
    if switch.time < timeline_start_time or switch.time > timeline_stop_time:
        continue
    ax.axvline(switch.time, color='red' if switch.mode == 'hi' else 'green', linestyle='--', linewidth=1.5)
    # ax.text(switch.time, len(name_to_id) - 0.5, f"Mode: {switch.mode}", rotation=90,
    #         verticalalignment='bottom', horizontalalignment='right' if switch.mode == 'hi' else 'left',
    #         color='red' if switch.mode == 'hi' else 'green')

# draw lines to separate swimlanes
for lane in range(len(name_to_id)):
    ax.axhline(lane + 0.5, color='black', linewidth=0.5, linestyle='--')

ax.set_yticks(list(name_to_id.values()))

# replace labels with friendly names if available
friendly_labels = [name_to_friendly.get(name, name) for name in name_to_id.keys()]
ax.set_yticklabels(friendly_labels)
# ax.set_yticklabels(list(name_to_id.keys()))
ax.set_xlabel("Time (s)")
# ax.set_title("Swimlane Plot of Node Executions")
plt.tight_layout()
plt.savefig("switch_example.pdf")
plt.show()

In [ ]:
import numpy as np
def get_times_for_node(node_name: str, selected_records = records):
    times = []
    for record in selected_records:
        if record.node_name == node_name:
            times.append((record.start_time, record.end_time, record.cpu_time))
    times = np.array(times) / 1000
    return times

records_in_lo = [r for r in records if r.mode == 'lo']
records_in_hi = [r for r in records if r.mode == 'hi']

print(f"LO mode statistics: ({len(records_in_lo)} records)")

for node in name_to_id.keys():
    times = get_times_for_node(node, records_in_lo)
    if len(times) == 0:
        continue
    print(f"Node: {node}:")
    print(f"\tMax CPU Time (ms): {np.max([t[2] for t in times])}, Avg CPU Time (ms): {np.mean([t[2] for t in times])}")

print()
print()

print(f"HI mode statistics: ({len(records_in_hi)} records)")
for node in name_to_id.keys():
    times = get_times_for_node(node, records_in_hi)
    if len(times) == 0:
        continue
    print(f"Node: {node}:")
    print(f"\tMax CPU Time (ms): {np.max([t[2] for t in times])}, Avg CPU Time (ms): {np.mean([t[2] for t in times])}")



In [ ]:
# calculate percent time spent in hi mode
total_time = records[-1].end_time - records[0].start_time
# hi_time = sum(record.end_time - record.start_time for record in records if record.mode == 'hi')
# calculate from mode switches!
hi_time = 0.0
last_switch_time = records[0].start_time
last_mode = 'lo'
for switch in mode_switches:
    if last_mode == 'hi':
        hi_time += switch.time - last_switch_time
    last_switch_time = switch.time
    last_mode = switch.mode
# account for time after last switch
if last_mode == 'hi':
    hi_time += records[-1].end_time - last_switch_time
hi_percentage = (hi_time / total_time) * 100
print(f"Total time: {total_time:.2f} s, HI mode time: {hi_time:.2f} s, HI mode percentage: {hi_percentage:.2f} %")

In [ ]:
# what were the HI spans, sorted by duration?
# Identify contiguous HI intervals based on mode_switches
hi_intervals = []
current_mode = 'lo'
# starting reference time: earliest record start
last_switch_time = records[0].start_time if records else 0.0
for ms in mode_switches:
    # when leaving HI, record interval
    if current_mode == 'hi' and ms.mode != 'hi':
        hi_intervals.append((last_switch_time, ms.time))
    # when entering HI, mark start
    if current_mode != 'hi' and ms.mode == 'hi':
        last_switch_time = ms.time
    elif current_mode == 'hi' and ms.mode == 'hi':
        # consecutive hi switches (if any) just reset start to latest
        last_switch_time = ms.time
    current_mode = ms.mode
# account for tail interval if still in HI at end
if current_mode == 'hi' and records:
    hi_intervals.append((last_switch_time, records[-1].end_time))

if not hi_intervals:
    print("No HI intervals found.")
else:
    # Build list with durations
    intervals_with_durations = [(start, end, end - start) for start, end in hi_intervals]
    # Sort descending by duration
    intervals_sorted = sorted(intervals_with_durations, key=lambda x: x[2], reverse=True)
    # Summary statistics
    durations = [d for _, _, d in intervals_sorted]
    total_hi_time = sum(durations)
    longest = intervals_sorted[0]
    print(f"Total HI intervals: {len(intervals_sorted)}")
    print(f"Aggregate HI time: {total_hi_time:.6f} s")
    print(f"Longest HI span: {longest[2]:.6f} s (from {longest[0]:.6f} to {longest[1]:.6f})")
    print()
    print("HI intervals sorted by duration (desc):")
    for idx, (start, end, dur) in enumerate(intervals_sorted, 1):
        print(f"#{idx:02d}: duration={dur:.6f} s start={start:.6f} end={end:.6f}")